<a href="https://colab.research.google.com/github/vzavala/DoE/blob/main/Python/ch5_heat_capacity_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Additional example: recursive Bayesian estimation of the heat-capacity model.**
With a Gaussian prior $\theta\sim\mathcal{N}(\bar{\theta},\Sigma_\theta)$ the Bayes estimate solves

$$\min_\theta\ \frac{1}{2}(\mathbf{y}-\mathbf{X}\theta)^T\Sigma^{-1}(\mathbf{y}-\mathbf{X}\theta)+\frac{1}{2}(\theta-\bar{\theta})^T\Sigma_\theta^{-1}(\theta-\bar{\theta}),$$

and the posterior covariance is $(\mathbf{X}^T\Sigma^{-1}\mathbf{X}+\Sigma_\theta^{-1})^{-1}$. Processing the data
in batches, the posterior of one batch becomes the prior of the next.
Python names: `theta_bar`, `Sigma_theta`, `Sigma` (noise covariance), `theta_hat`.

**Note:** This notebook reproduces Example 5.16 (Section 5.5) of *Statistics for Chemical Engineers*: updating the heat-capacity estimates one batch of data at a time.

**The concept.** Setting the gradient of the Bayes objective to zero gives a linear system,

$$\left(\mathbf{X}^T\Sigma^{-1}\mathbf{X}+\Sigma_\theta^{-1}\right)\hat{\theta}=\mathbf{X}^T\Sigma^{-1}\mathbf{y}+\Sigma_\theta^{-1}\bar{\theta}.$$

The matrix on the left is the **total information**: information in the data, $\mathbf{X}^T\Sigma^{-1}\mathbf{X}$, plus information in the prior, $\Sigma_\theta^{-1}$. Its inverse is the posterior covariance. For independent batches the data information is a sum, $\sum_k\mathbf{X}_k^T\Sigma_k^{-1}\mathbf{X}_k$, so we can add one batch at a time and use each posterior as the next prior.

In [1]:
import numpy as np

theta = np.array([6.190, 2.923e-3, -7.052e-7])
x = np.repeat([300.0, 600.0, 900.0], 20)
S = len(x)
X = np.column_stack([np.ones(S), x, x**2])
np.random.seed(0)
sigma = 0.1
y = X @ theta + np.random.normal(0, sigma, S)
idx = np.random.permutation(S)
X, y = X[idx], y[idx]

**From equation to code.** `A` is the total information matrix and `b` the right-hand side of the linear system. `np.linalg.solve(Sigma, X)` computes $\Sigma^{-1}\mathbf{X}$ without forming the inverse.

In [2]:
def bayes_estimate(X, y, Sigma, theta_bar, Sigma_theta):
    """Minimizer of the Bayes objective (closed form for a linear model) and posterior covariance."""
    A = X.T @ np.linalg.solve(Sigma, X) + np.linalg.inv(Sigma_theta)
    b = X.T @ np.linalg.solve(Sigma, y) + np.linalg.solve(Sigma_theta, theta_bar)
    return np.linalg.solve(A, b), np.linalg.inv(A)

theta_bar, Sigma_theta = np.ones(3), np.eye(3)
theta_hat, Sigma_post = bayes_estimate(X, y, sigma**2 * np.eye(S), theta_bar, Sigma_theta)
print("all data at once:  theta_hat =", np.array2string(theta_hat, precision=4))

all data at once:  theta_hat = [ 6.2546e+00  2.9045e-03 -8.1641e-07]


Recursive estimation with batches of 10 experiments (6 cycles, as in the slides):

In [3]:
batch = 10
for k in range(S // batch):
    rows = slice(k * batch, (k + 1) * batch)
    theta_bar, Sigma_theta = bayes_estimate(X[rows], y[rows], sigma**2 * np.eye(batch), theta_bar, Sigma_theta)
    print(f"after batch {k+1}: theta_hat = {np.array2string(theta_bar, precision=4)},  "
          f"SD = {np.array2string(np.sqrt(np.diag(Sigma_theta)), precision=2)}")

after batch 1: theta_hat = [ 5.7543e+00  4.5517e-03 -2.1165e-06],  SD = [2.43e-01 9.15e-04 7.49e-07]
after batch 2: theta_hat = [ 6.0420e+00  3.7342e-03 -1.4977e-06],  SD = [1.76e-01 6.87e-04 5.66e-07]
after batch 3: theta_hat = [ 6.1697e+00  3.1297e-03 -9.5508e-07],  SD = [1.47e-01 5.45e-04 4.42e-07]
after batch 4: theta_hat = [ 6.1788e+00  3.2053e-03 -1.0496e-06],  SD = [1.22e-01 4.64e-04 3.81e-07]
after batch 5: theta_hat = [ 6.1899e+00  3.1344e-03 -9.9731e-07],  SD = [1.07e-01 4.06e-04 3.35e-07]
after batch 6: theta_hat = [ 6.2546e+00  2.9045e-03 -8.1641e-07],  SD = [9.70e-02 3.67e-04 3.03e-07]


**Check the math.** The recursive result equals the result with all the data at once, for the estimate and for its covariance.

In [4]:
print("same estimate:  ", np.allclose(theta_bar, theta_hat, rtol=1e-4))
print("same covariance:", np.allclose(Sigma_theta, Sigma_post, rtol=1e-4))

same estimate:   True
same covariance: True


**Note:** As in the book, after the 6th update the covariance equals the one obtained with the full data set: the posterior of one step is the prior of the next.